In [1]:
import pandas as pd
import json

In [6]:
import os

def rename_images(folder_path):
    # Get all files in the folder
    files = os.listdir(folder_path)
    
    # Filter only image files by common extensions
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp'}
    images = [f for f in files if os.path.splitext(f)[1].lower() in image_extensions]
    
    # Sort for consistent ordering
    images.sort()
    
    # Rename files
    for i, filename in enumerate(images, start=1):
        old_path = os.path.join(folder_path, filename)
        ext = os.path.splitext(filename)[1]
        new_name = f"good{i}.jpg"
        new_path = os.path.join(folder_path, new_name)
        
        os.rename(old_path, new_path)
        print(f"Renamed: {filename} -> {new_name}")

# Example usage
if __name__ == "__main__":
    folder = f"../data/good"  # change this to your folder
    rename_images(folder)


Renamed: goodd1.jpg -> good1.jpg
Renamed: goodd10.jpg -> good2.jpg
Renamed: goodd100.jpg -> good3.jpg
Renamed: goodd11.jpg -> good4.jpg
Renamed: goodd12.jpg -> good5.jpg
Renamed: goodd13.jpg -> good6.jpg
Renamed: goodd14.jpg -> good7.jpg
Renamed: goodd15.jpg -> good8.jpg
Renamed: goodd16.jpg -> good9.jpg
Renamed: goodd17.jpg -> good10.jpg
Renamed: goodd18.jpg -> good11.jpg
Renamed: goodd19.jpg -> good12.jpg
Renamed: goodd2.jpg -> good13.jpg
Renamed: goodd20.jpg -> good14.jpg
Renamed: goodd21.jpg -> good15.jpg
Renamed: goodd22.jpg -> good16.jpg
Renamed: goodd23.jpg -> good17.jpg
Renamed: goodd24.jpg -> good18.jpg
Renamed: goodd25.jpg -> good19.jpg
Renamed: goodd26.jpg -> good20.jpg
Renamed: goodd27.jpg -> good21.jpg
Renamed: goodd28.jpg -> good22.jpg
Renamed: goodd29.jpg -> good23.jpg
Renamed: goodd3.jpg -> good24.jpg
Renamed: goodd30.jpg -> good25.jpg
Renamed: goodd31.jpg -> good26.jpg
Renamed: goodd32.jpg -> good27.jpg
Renamed: goodd33.jpg -> good28.jpg
Renamed: goodd34.jpg -> good29.

In [7]:
import numpy as np

def get_shoulder_width(landmarks):
  left_shoulder = np.array([landmarks[11].x, landmarks[11].y])
  right_shoulder = np.array([landmarks[12].x, landmarks[12].y])
  return np.linalg.norm(left_shoulder - right_shoulder)

def scale_variance(landmarks, factor):
  left_shoulder = np.array([landmarks[11].x, landmarks[11].y])
  right_shoulder = np.array([landmarks[12].x, landmarks[12].y])
  center = (left_shoulder + right_shoulder) / 2

  shoulder_width = get_shoulder_width(landmarks)
  
  min_x = float('inf')
  max_x = float('-inf')
  min_y = float('inf')
  max_y = float('-inf')
  for landmark in landmarks:
    point = np.array([landmark.x, landmark.y])
    normalized_point = center + factor * (point - center)
    landmark.x = normalized_point[0]
    landmark.y = normalized_point[1]

    min_x = min(min_x, normalized_point[0])
    max_x = max(max_x, normalized_point[0])

    min_y = min(min_y, normalized_point[1])
    max_y = max(max_y, normalized_point[1])

  # center = (np.array([landmarks[7].x, landmarks[7].y]) + np.array([landmarks[8].x, landmarks[8].y])) / 2
  # for landmark in landmarks:
  #   point = np.array([landmark.x, landmark.y])
  #   normalized_point = (point - center) / shoulder_width
  #   landmark.x = normalized_point[0]
  #   landmark.y = normalized_point[1]

  # for landmark in landmarks:
  #   landmark.x = (landmark.x - min_x) / (max_x - min_x)
  #   landmark.y = (landmark.y - min_y) / (max_y - min_y)

In [8]:
import cv2
import mediapipe as mp
import pandas as pd
import os
import random

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.7,
                       min_tracking_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

landmark_names = [
    'nose', 'left_eye_inner', 'left_eye', 'left_eye_outer', 'right_eye_inner', 'right_eye', 'right_eye_outer',
    'left_ear', 'right_ear', 'mouth_left', 'mouth_right', 'left_shoulder', 'right_shoulder', 'left_elbow',
    'right_elbow', 'left_wrist', 'right_wrist', 'left_pinky', 'right_pinky', 'left_index', 'right_index',
    'left_thumb', 'right_thumb', 'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle',
    'left_heel', 'right_heel', 'left_foot_index', 'right_foot_index'
]

data = []

min_variance = 0.5
max_variance = 1.5

results = None
for i in range(1, 101):
    paths = [f"../data/bad/bad{i}.jpg", f"../data/good/good{i}.jpg"]
    for k in range(2):
      path = paths[k]
      image = cv2.imread(path)
      if image is None:
        print(i)
      rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
      results = pose.process(rgb_image)

      for j in range(10):
        landmarks = []
        for name in landmark_names:
            idx = mp_pose.PoseLandmark[name.upper()].value
            lm = results.pose_landmarks.landmark[idx]
            landmarks.append([lm.x, lm.y, lm.z, lm.visibility])

        random_variance = random.uniform(min_variance, max_variance)
        scale_variance(results.pose_landmarks.landmark, random_variance)

        data.append([landmarks, k])

# for i in range(1, 51):
#     path = f"../data/bad/bad{i}.jpg"
#     image = cv2.imread(path)
#     rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#     results = pose.process(rgb_image)

#     random_variance = random.uniform(min_variance, max_variance)
#     scale_variance(results.pose_landmarks.landmark, random_variance)

#     landmarks = []
#     for name in landmark_names:
#         idx = mp_pose.PoseLandmark[name.upper()].value
#         lm = results.pose_landmarks.landmark[idx]
#         landmarks.append([lm.x, lm.y, lm.z, lm.visibility])
#     data.append([landmarks, 0])
len(data)

2000

In [9]:
import json

with open('../data/data6.json', 'w') as f:
    json.dump(data, f)

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.framework.formats import landmark_pb2

# Initialize MediaPipe Hands
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.7,
                       min_tracking_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils


frame_1 = np.zeros((480, 640, 3), dtype=np.uint8)
frame_2 = np.zeros((480, 640, 3), dtype=np.uint8)

path = f"../data/good/good{11}.jpg"
image = cv2.imread(path)
rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
results = pose.process(rgb_image)

# Draw with filtered landmarks and filtered connections
mp_drawing.draw_landmarks(frame_1, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

print(results.pose_landmarks.landmark[0])
scale_variance(results.pose_landmarks.landmark, 1.5)
print(results.pose_landmarks.landmark[0])

mp_drawing.draw_landmarks(
        frame_2, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

while True:
    # Display the frame
    cv2.imshow('MediaPipe Hand Tracking', frame_1)

    # Exit if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.destroyAllWindows()
        break

while True:   
    cv2.imshow('MediaPipe Hand Tracking', frame_2)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.destroyAllWindows()
        break

x: 0.41344589
y: 0.376931548
z: -0.358267576
visibility: 0.999893904

x: 0.370055526
y: 0.213787541
z: -0.358267576
visibility: 0.999893904



[1, 1, 1] # <- nose

[[1, 1, 1], [2, 2, 2], ...] # <- nose, left eye inner, left heel, etc

[[[1, 1, 1], [2, 2, 2], ...], [[1, 1, 1], [2, 2, 2], ...], [[1, 1, 1], [2, 2, 2], ...]] # <- each individual array is passed to the model

In [11]:
import torch
import torch.nn as nn

HIDDEN_UNITS = 256
model = nn.Sequential(
  nn.Linear(132, HIDDEN_UNITS),
  nn.ReLU(),
  nn.Linear(HIDDEN_UNITS, HIDDEN_UNITS),
  nn.ReLU(),
  nn.Linear(HIDDEN_UNITS, 2)
).to(device="cuda")

model.load_state_dict(torch.load("../state_dicts/model_2_state_dict.pth"))

<All keys matched successfully>

In [12]:
import cv2
import numpy as np
import mediapipe as mp

def logits_to_labels(y_logits):
  y_pred_probs = torch.softmax(y_logits, dim=1)
  y_preds = torch.argmax(y_pred_probs, dim=1)
  return y_preds

# Initialize MediaPipe Hands
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.7,
                       min_tracking_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

# Open webcam
cap = cv2.VideoCapture(0)

landmarks = []
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Flip the frame horizontally for a mirrored view
    frame = cv2.flip(frame, 1)

    # Convert the BGR image to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process the frame with MediaPipe Hands
    results = pose.process(rgb_frame)

    if results.pose_landmarks == None:
        continue
    # normalize_landmarks_to_shoulders(results.pose_landmarks.landmark)

    landmarks = []
    for name in landmark_names:
        idx = mp_pose.PoseLandmark[name.upper()].value
        lm = results.pose_landmarks.landmark[idx]
        # landmarks.append([lm.x, lm.y, lm.z, lm.visibility])
        landmarks.append(lm.x)
        landmarks.append(lm.y)
        landmarks.append(lm.z)
        landmarks.append(lm.visibility)

    landmarks = torch.tensor(landmarks).to(device="cuda")
    # Let's get some raw outputs of our model (logits)
    model.eval()
    with torch.inference_mode():
        y_logits = model(landmarks)
    label = logits_to_labels(y_logits=y_logits.unsqueeze(0))

    color = (0, 255, 0) if label == 1 else (0, 0, 255)
    # Draw hand landmarks if detected
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, landmark_drawing_spec=mp_drawing.DrawingSpec(color=color, thickness=2, circle_radius=4))

    # Display the frame
    cv2.imshow('MediaPipe Hand Tracking', frame)

    # Exit if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
pose.close()
cap.release()
cv2.destroyAllWindows()

In [ ]:
data = []

for i, landmark in enumerate(landmarks[0].landmark):
  data.append([landmark.x, landmark.y, landmark.visibility])
  print(landmark)

x: 0.696406364
y: 0.301912576
z: -1.22071028
visibility: 0.999789417

x: 0.711333156
y: 0.238704681
z: -1.14707303
visibility: 0.999579608

x: 0.724056602
y: 0.239943981
z: -1.14733899
visibility: 0.999554098

x: 0.736935854
y: 0.241262913
z: -1.14732873
visibility: 0.99942553

x: 0.663240969
y: 0.235795736
z: -1.17617965
visibility: 0.999725282

x: 0.639987
y: 0.235771045
z: -1.17605674
visibility: 0.999782383

x: 0.615409
y: 0.237026826
z: -1.17661619
visibility: 0.999787748

x: 0.735137403
y: 0.269437909
z: -0.666463673
visibility: 0.999457657

x: 0.559208632
y: 0.271661639
z: -0.79201591
visibility: 0.999883175

x: 0.713557363
y: 0.371393085
z: -1.03803921
visibility: 0.999814212

x: 0.655462563
y: 0.369271964
z: -1.07383895
visibility: 0.999927759

x: 0.844586253
y: 0.62008816
z: -0.308202654
visibility: 0.996193647

x: 0.400108039
y: 0.62043488
z: -0.54830265
visibility: 0.998165429

x: 0.954881489
y: 0.947976708
z: -0.234480813
visibility: 0.328588575

x: 0.329506248
y: 1.009743

In [ ]:
data[0]

[0.696406364440918, 0.3019125759601593, 0.9997894167900085]